In [0]:
df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.includeExistingFiles", "true")
    .option("header", "true")
    .load(containername)
    .withColumn("source_file", col("_metadata.file_path"))
    .withColumn("ingesttime", current_timestamp())
)


In [0]:
display(
    df,
    checkpointLocation="abfss://raw@stggen2accountchenna.dfs.core.windows.net/checkpoints/bronze_display"
)

query = (
    df.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        "abfss://raw@stggen2accountchenna.dfs.core.windows.net/checkpoints/bronze_table"
    )
    .trigger(availableNow=True)
    .toTable(bronze_table_path)
)

query.awaitTermination()
